In [41]:
import json
import os
from typing import List, Dict, Any
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

# 原始文本
text = """这是一份江南农村商业银行的业务委托书，包含了委托人的基本信息、收款人的信息、汇款金额、支付密码、用途等详细信息。
 % 类别 : 业务委托书-处理  %
关键字段：委托日期: 2021年1月M日D | 金额(大写): 壹拾肆万元整 | 金额(小写): ￥14000000 | 支付密码: 72610176.6021.3415 | 用途: 货款 | 汇款方式: 普通 | 委托人签名: 方徐印春
这是一份银行汇款委托书，详细记录了汇款人的信息、收款人的信息、汇款金额、汇款用途以及汇款的具体细节。"""

# 定义分隔符
separators = ["%" ,", " ,"\n\n", "\n", "。", "；", " | ", " "]

# 初始化分割器
splitter = RecursiveCharacterTextSplitter(
    chunk_size=20,  # 最大每个块的字符数
    chunk_overlap=5,  # 每个块之间的重叠
    separators=separators  # 定义分隔符
)

# 执行切分
chunks = splitter.split_text(text)

# 输出切分后的块
for i, chunk in enumerate(chunks, 1):
    print(f"文本块 {i}:")
    print(chunk)
    print("-" * 30)


文本块 1:
这是一份江南农村商业银行的业务委托书，包含了委托人的基本信息、收款人的信息、汇款金额、支付密码、用途等详细信息
------------------------------
文本块 2:
。
------------------------------
文本块 3:
% 类别 : 业务委托书-处理
------------------------------
文本块 4:
%
------------------------------
文本块 5:
关键字段：委托日期:
------------------------------
文本块 6:
2021年1月M日D
------------------------------
文本块 7:
| 金额(大写): 壹拾肆万元整
------------------------------
文本块 8:
| 金额(小写): ￥14000000
------------------------------
文本块 9:
| 支付密码:
------------------------------
文本块 10:
72610176.6021.3415
------------------------------
文本块 11:
| 用途: 货款 | 汇款方式: 普通
------------------------------
文本块 12:
| 委托人签名: 方徐印春
------------------------------
文本块 13:

这是一份银行汇款委托书，详细记录了汇款人的信息、收款人的信息、汇款金额、汇款用途以及汇款的具体细节
------------------------------
文本块 14:
。
------------------------------


In [6]:
import json
import os
from typing import List, Dict, Any
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
import uuid
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore  # 新增


# ==================== 配置 ====================
JSONL_BASE = "/data/postgraduates/2024/chenjiarui/Model/Agent/script/rag/data/knowledge_base.jsonl"
JSONL_CLASS = "/data/postgraduates/2024/chenjiarui/Model/Agent/script/rag/data/knowledge_class.jsonl"
PERSIST_DIR = "../data/chroma_db_final" 

EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
BASE_URL = "https://api.siliconflow.cn/v1"
API_KEY = os.getenv("SILICONFLOW_API_KEY", "sk-tgprnspwkhliprfcuobqpfiiwjawxkgaldpfkjtovpfudpmf")

CHUNK_SIZE = 600
CHUNK_OVERLAP = 120
TOP_K = 5


class VectorStoreManager:
    def __init__(self, persist_directory: str = PERSIST_DIR):
        os.makedirs(persist_directory, exist_ok=True)
        print("正在初始化 Qwen Embedding 模型...")
        self.embeddings = OpenAIEmbeddings(
            model=EMBEDDING_MODEL,
            base_url=BASE_URL,
            api_key=API_KEY,
        )
        self.vector_store = Chroma(
            persist_directory=persist_directory,
            embedding_function=self.embeddings,
            collection_name="doc_rag",
        )

        # 大块切分器（原逻辑）
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n\n", "\n", "。", "；", "，", " | ", " "],
        )

        # 小块切分器（用于精确检索）
        self.child_splitter = RecursiveCharacterTextSplitter(
            chunk_size=80,   # 更小，适合你的字段长度
            chunk_overlap=15,
            separators=["。", "，", " | ", " "]
        )

        # ParentDocumentRetriever + InMemoryStore
        self.docstore = InMemoryStore()  # 存储大块
        self.retriever = ParentDocumentRetriever(
            vectorstore=self.vector_store,
            docstore=self.docstore,
            child_splitter=self.child_splitter,
            # parent_splitter=self.text_splitter,  # 可选：大块也切
        )

        print("向量库初始化完成（已启用 ParentDocumentRetriever）")

    # ------------------- 加载 label 解释 -------------------
    def _load_label_intro(self) -> Dict[str, str]:
        intro = {}
        if not os.path.exists(JSONL_CLASS):
            print(f"警告: 未找到 {JSONL_CLASS}，将跳过 label 解释")
            return intro
        with open(JSONL_CLASS, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line: continue
                try:
                    obj = json.loads(line)
                    k, v = list(obj.items())[0]
                    intro[k] = v
                except:
                    continue
        print(f"成功加载 {len(intro)} 条 label 解释")
        return intro

    # ------------------- 加载主文档 -------------------
    def _load_documents(self) -> List[Document]:
        docs: List[Document] = []
        seen = set()
        label_intro = self._load_label_intro()

        if not os.path.exists(JSONL_BASE):
            raise FileNotFoundError(f"未找到主文件: {JSONL_BASE}")

        with open(JSONL_BASE, "r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, 1):
                line = line.strip()
                if not line: continue
                try:
                    data = json.loads(line)
                    vlm = data.get("VLM_text", {})
                    if isinstance(vlm, str):
                        vlm = json.loads(vlm)

                    # 提取关键字段
                    key_fields = vlm.get("key_fields", {})
                    summary = vlm.get("content_summary", "").strip()

                    # 构建完整内容：摘要 + 关键字段
                    field_text = " | ".join(
                        f"{k}: {v}" for k, v in key_fields.items() if v and str(v).strip()
                    )
                    content = f"{summary}"
                    if field_text:
                        content += f"\n关键字段：{field_text}"

                    if not content.strip():
                        continue

                    # 元数据
                    label = data.get("label", "")
                    source = data.get("image_path", "").split("data/")[-1]
                    metadata = {
                        "file_id": str(uuid.uuid4()),
                        "source": source,
                        "label": label,
                        "label_introduction": label_intro.get(label, '未知文档类型'),
                        "key_fields_json": json.dumps(key_fields, ensure_ascii=False),
                        "line": line_no,
                    }
                    print(source)
                    print(content, "\n")

                    # 去重
                    key = (source, line_no)
                    if key in seen: continue
                    seen.add(key)

                    # 关键：大块用原始 content
                    docs.append(Document(page_content=content, metadata=metadata))

                except Exception as e:
                    print(f"[第 {line_no} 行] 解析失败: {e}")

        print(f"共加载 {len(docs)} 条有效文档")
        return docs

    # ------------------- 添加文档（增量） -------------------
    def add_documents(self) -> int:
        raw_docs = self._load_documents()
        if not raw_docs:
            raise ValueError("没有加载到任何文档")

        # 使用 ParentDocumentRetriever 自动切小块 + 存大块
        self.retriever.add_documents(raw_docs)
        print(f"成功写入向量库（小块用于检索，大块用于返回）")
        return len(raw_docs)  # 返回大块数量

    # ------------------- 检索 -------------------
    def search(self, query: str, top_k: int = TOP_K) -> List[Dict[str, Any]]:
        print(f"\n执行检索: {query}")
        # 使用 retriever.invoke 返回大块
        results = self.retriever.invoke(query, limit=top_k)

        formatted = []
        for doc in results:
            # 模拟 score（可通过 similarity_search 再算）
            formatted.append({
                "content": doc.page_content,
                "metadata": doc.metadata,
                "similarity": 0.0  # 可选：加相似度
            })

        print(f"返回 {len(formatted)} 条完整凭证结果")
        return formatted


# ==================== 运行入口 ====================
def main():
    print("开始构建 RAG 向量库...")
    manager = VectorStoreManager()

    # 构建/更新向量库
    manager.add_documents()

    # 示例检索
    print("\n" + "="*60)
    print("示例检索（效果展示）")
    print("="*60)

    test_queries = [
        "日期: 2021年01月25日",
        "转账原因: 季晓玲12月伙食",
        "金额(小写): ￥110000"
    ]

    for q in test_queries:
        print(f"\n> 查询: {q}")
        results = manager.search(q, top_k=3)
        for i, r in enumerate(results, 1):
            print(f"\n  [{i}] 完整凭证")
            print("=" * 80)
            
            # 1. 完整 content（不截断）
            print("  内容：")
            print(f"    {r['content']}")
            print()
            
            # 2. 完整 metadata（格式化 JSON）
            print("  元数据：")
            import json
            metadata_str = json.dumps(r['metadata'], ensure_ascii=False, indent=4)
            print(f"    {metadata_str}")
            
            # 3. 来源摘要
            source_file = os.path.basename(r['metadata'].get('source', 'unknown'))
            label = r['metadata'].get('label', '未知')
            print(f"\n  来源: {source_file} | 类型: {label}")
            print("-" * 80)


if __name__ == "__main__":
    main()

开始构建 RAG 向量库...
正在初始化 Qwen Embedding 模型...
向量库初始化完成（已启用 ParentDocumentRetriever）
成功加载 10 条 label 解释
图片示例/营业执照-处理/image_1.jpg
这是一份中国企业的营业执照副本，包含了企业的基本信息、注册资本、成立日期、营业期限和经营范围等详细信息。
关键字段：统一社会信用代码: ********** | 注册资本: 1000万元整 | 成立日期: 2018年12月05日 | 营业期限: 2018年12月05日至****** | 经营范围: 计算机软件、互联网领域的技术开发、技术转让、技术咨询、技术服务；计算机系统集成及运行维护，数据处理及存储服务。（依法须经批准的项目，经相关部门批准后方可开展经营活动） 

图片示例/营业执照-处理/image_2.jpg
这是一份中国企业的营业执照副本，包含了企业的基本信息、经营范围、注册资本、成立日期、营业期限、住所以及登记机关和日期。
关键字段：统一社会信用代码: 320581666202101270316 | 经营范围: 清洁用除尘装置、机器传动带、风力发电设备、蒸汽机、注塑机、包装机、压花机、包饺子机、染色机、起绒毛机、整理机、磁辊印花机、智能机械手、电子机械设备、激光机械设备、镜像机械设备、分料集成设备、中控机械设备、远程通讯设备、汽车顶棚复合材料、电气设备、通用设备、纺织机械设备制造、销售、安装、维护、改造及售后服务；非标机电设备及配件制造与销售。（依法须经批准的项目，经相关部门批准后方可开展经营活动） | 注册资本: 1000万元整 | 成立日期: 2016年06月23日 | 营业期限: 2016年06月23日至****** | 登记机关: 常熟市行政审批局 | 登记日期: 2021年01月27日 

图片示例/营业执照-处理/image_3.jpg
这是一份中国企业的营业执照，包含了企业的基本信息、注册资本、成立日期、营业期限以及经营范围等详细信息。
关键字段：注册资本: 6000万元整 | 成立日期: 2014年03月25日 | 营业期限: 2014年03月25日至2024年03月24日 | 经营范围: 有线电视分配器、光纤适配器、光纤连接器、光发射机配件、接插件、铜制品的研发、生产，有线电视分配器、线缆、天

In [1]:
import json
import os
from typing import List, Dict, Any
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
import uuid
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore  # 新增


# ==================== 配置 ====================
JSONL_BASE = "/data/postgraduates/2024/chenjiarui/Model/Agent/script/rag/data/knowledge_base.jsonl"
JSONL_CLASS = "/data/postgraduates/2024/chenjiarui/Model/Agent/script/rag/data/knowledge_class.jsonl"
PERSIST_DIR = "../data/chroma_db_final" 

EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
BASE_URL = "https://api.siliconflow.cn/v1"
API_KEY = os.getenv("SILICONFLOW_API_KEY", "sk-tgprnspwkhliprfcuobqpfiiwjawxkgaldpfkjtovpfudpmf")

CHUNK_SIZE = 600
CHUNK_OVERLAP = 120
TOP_K = 5


class VectorStoreManager:
    def __init__(self, persist_directory: str = PERSIST_DIR):
        os.makedirs(persist_directory, exist_ok=True)
        print("正在初始化 Qwen Embedding 模型...")
        self.embeddings = OpenAIEmbeddings(
            model=EMBEDDING_MODEL,
            base_url=BASE_URL,
            api_key=API_KEY,
        )
        self.vector_store = Chroma(
            persist_directory=persist_directory,
            embedding_function=self.embeddings,
            collection_name="doc_rag",
        )

        # 大块切分器（原逻辑）
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n\n", "\n", "。", "；", "，", " | ", " "],
        )

        # 小块切分器（用于精确检索）
        self.child_splitter = RecursiveCharacterTextSplitter(
            chunk_size=80,   # 更小，适合你的字段长度
            chunk_overlap=15,
            separators=["。", "，", " | ", " "]
        )

        # ParentDocumentRetriever + InMemoryStore
        self.docstore = InMemoryStore()  # 存储大块
        self.retriever = ParentDocumentRetriever(
            vectorstore=self.vector_store,
            docstore=self.vector_store,
            child_splitter=self.child_splitter,
            parent_splitter=self.text_splitter,  # 可选：大块也切
        )

        print("向量库初始化完成（已启用 ParentDocumentRetriever）")

    # ------------------- 加载 label 解释 -------------------
    def _load_label_intro(self) -> Dict[str, str]:
        intro = {}
        if not os.path.exists(JSONL_CLASS):
            print(f"警告: 未找到 {JSONL_CLASS}，将跳过 label 解释")
            return intro
        with open(JSONL_CLASS, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line: continue
                try:
                    obj = json.loads(line)
                    k, v = list(obj.items())[0]
                    intro[k] = v
                except:
                    continue
        print(f"成功加载 {len(intro)} 条 label 解释")
        return intro

    # ------------------- 加载主文档 -------------------
    def _load_documents(self) -> List[Document]:
        docs: List[Document] = []
        seen = set()
        label_intro = self._load_label_intro()

        if not os.path.exists(JSONL_BASE):
            raise FileNotFoundError(f"未找到主文件: {JSONL_BASE}")

        with open(JSONL_BASE, "r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, 1):
                line = line.strip()
                if not line: continue
                try:
                    data = json.loads(line)
                    vlm = data.get("VLM_text", {})
                    if isinstance(vlm, str):
                        vlm = json.loads(vlm)

                    # 提取关键字段
                    key_fields = vlm.get("key_fields", {})
                    summary = vlm.get("content_summary", "").strip()

                    # 构建完整内容：摘要 + 关键字段
                    field_text = " | ".join(
                        f"{k}: {v}" for k, v in key_fields.items() if v and str(v).strip()
                    )
                    content = f"{summary}"
                    if field_text:
                        content += f"\n关键字段：{field_text}"

                    if not content.strip():
                        continue

                    # 元数据
                    label = data.get("label", "")
                    source = data.get("image_path", "").split("data/")[-1]
                    metadata = {
                        "file_id": str(uuid.uuid4()),
                        "source": source,
                        "label": label,
                        "label_introduction": label_intro.get(label, '未知文档类型'),
                        "key_fields_json": json.dumps(key_fields, ensure_ascii=False),
                        "line": line_no,
                    }
                    print(source)
                    print(content, "\n")

                    # 去重
                    key = (source, line_no)
                    if key in seen: continue
                    seen.add(key)

                    # 关键：大块用原始 content
                    docs.append(Document(page_content=content, metadata=metadata))

                except Exception as e:
                    print(f"[第 {line_no} 行] 解析失败: {e}")

        print(f"共加载 {len(docs)} 条有效文档")
        return docs

    # ------------------- 添加文档（增量） -------------------
    def add_documents(self) -> int:
        raw_docs = self._load_documents()
        if not raw_docs:
            raise ValueError("没有加载到任何文档")

        # 使用 ParentDocumentRetriever 自动切小块 + 存大块
        self.retriever.add_documents(raw_docs)
        print(f"成功写入向量库（小块用于检索，大块用于返回）")
        return len(raw_docs)  # 返回大块数量

    # ------------------- 检索 -------------------
    def search(self, query: str, top_k: int = TOP_K) -> List[Dict[str, Any]]:
        print(f"\n执行检索: {query}")
        # 使用 retriever.invoke 返回大块
        results = self.retriever.invoke(query, limit=top_k)

        formatted = []
        for doc in results:
            # 模拟 score（可通过 similarity_search 再算）
            formatted.append({
                "content": doc.page_content,
                "metadata": doc.metadata,
                "similarity": 0.0  # 可选：加相似度
            })

        print(f"返回 {len(formatted)} 条完整凭证结果")
        return formatted



def main():
    print("开始构建 RAG 向量库...")
    manager = VectorStoreManager()


    # 示例检索
    print("\n" + "="*60)
    print("示例检索（效果展示）")
    print("="*60)

    test_queries = [
        "日期: 2021年01月25日",
        "转账原因: 季晓玲12月伙食",
        "金额(小写): ￥110000"
    ]


    for q in test_queries:
        print(f"\n> 查询: {q}")
        results = manager.search(q, top_k=3)
        for i, r in enumerate(results, 1):
            print(f"\n  [{i}] 完整凭证")
            print("=" * 80)
            
            # 1. 完整 content（不截断）
            print("  内容：")
            print(f"    {r['content']}")
            print()
            
            # 2. 完整 metadata（格式化 JSON）
            print("  元数据：")
            import json
            metadata_str = json.dumps(r['metadata'], ensure_ascii=False, indent=4)
            print(f"    {metadata_str}")
            
            # 3. 来源摘要
            source_file = os.path.basename(r['metadata'].get('source', 'unknown'))
            label = r['metadata'].get('label', '未知')
            print(f"\n  来源: {source_file} | 类型: {label}")
            print("-" * 80)


if __name__ == "__main__":
    main()

开始构建 RAG 向量库...
正在初始化 Qwen Embedding 模型...
向量库初始化完成（已启用 ParentDocumentRetriever）

示例检索（效果展示）

> 查询: 日期: 2021年01月25日

执行检索: 日期: 2021年01月25日
返回 0 条完整凭证结果

> 查询: 转账原因: 季晓玲12月伙食

执行检索: 转账原因: 季晓玲12月伙食
返回 0 条完整凭证结果

> 查询: 金额(小写): ￥110000

执行检索: 金额(小写): ￥110000
返回 0 条完整凭证结果


In [3]:
# 修复版 VectorStoreManager（使用 LocalFileStore + pickle）
import json
import os
import uuid
import pickle
from typing import List, Dict, Any

from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain.storage import LocalFileStore


# ==================== 配置 ====================
JSONL_BASE = "/data/postgraduates/2024/chenjiarui/Model/Agent/script/rag/data/knowledge_base.jsonl"
JSONL_CLASS = "/data/postgraduates/2024/chenjiarui/Model/Agent/script/rag/data/knowledge_class.jsonl"
PERSIST_DIR = "../data/chroma_db_final"
PARENT_DIR = "../data/parent_docs" 

EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
BASE_URL = "https://api.siliconflow.cn/v1"
API_KEY = os.getenv("SILICONFLOW_API_KEY", "sk-tgprnspwkhliprfcuobqpfiiwjawxkgaldpfkjtovpfudpmf")

CHUNK_SIZE = 600
CHUNK_OVERLAP = 120
CHILD_CHUNK_SIZE = 80
CHILD_CHUNK_OVERLAP = 15
TOP_K = 5


class VectorStoreManager:
    def __init__(self, persist_directory: str = PERSIST_DIR, parent_dir: str = PARENT_DIR):
        os.makedirs(persist_directory, exist_ok=True)
        os.makedirs(parent_dir, exist_ok=True)
        print("正在初始化 Qwen Embedding 模型...")
        self.embeddings = OpenAIEmbeddings(
            model=EMBEDDING_MODEL,
            base_url=BASE_URL,
            api_key=API_KEY,
        )

        # 向量库（小块）
        self.vector_store = Chroma(
            persist_directory=persist_directory,
            embedding_function=self.embeddings,
            collection_name="doc_rag",
        )

        # 大文档存储（持久化）
        self.docstore = LocalFileStore(parent_dir)

        # 小块切分器
        self.child_splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHILD_CHUNK_SIZE,
            chunk_overlap=CHILD_CHUNK_OVERLAP,
            separators=["。", "，", " | ", " "],
        )

        print("向量库 + 大文档存储初始化完成")

    # ------------------- 加载 label -------------------
    def _load_label_intro(self) -> Dict[str, str]:
        intro = {}
        if not os.path.exists(JSONL_CLASS):
            print(f"警告: 未找到 {JSONL_CLASS}")
            return intro
        with open(JSONL_CLASS, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line: continue
                try:
                    obj = json.loads(line)
                    k, v = list(obj.items())[0]
                    intro[k] = v
                except:
                    continue
        return intro

    # ------------------- 加载文档 -------------------
    def _load_documents(self) -> List[Document]:
        docs: List[Document] = []
        seen = set()
        label_intro = self._load_label_intro()

        if not os.path.exists(JSONL_BASE):
            raise FileNotFoundError(f"未找到: {JSONL_BASE}")

        with open(JSONL_BASE, "r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, 1):
                line = line.strip()
                if not line: continue
                try:
                    data = json.loads(line)
                    vlm = data.get("VLM_text", {})
                    if isinstance(vlm, str):
                        vlm = json.loads(vlm)

                    key_fields = vlm.get("key_fields", {})
                    summary = vlm.get("content_summary", "").strip()

                    field_text = " | ".join(
                        f"{k}: {v}" for k, v in key_fields.items() if v and str(v).strip()
                    )
                    content = summary
                    if field_text:
                        content += f"\n关键字段：{field_text}"

                    if not content.strip():
                        continue

                    label = data.get("label", "")
                    source = data.get("image_path", "").split("data/")[-1]
                    file_id = str(uuid.uuid4())
                    metadata = {
                        "file_id": file_id,
                        "source": source,
                        "label": label,
                        "label_introduction": label_intro.get(label, "未知"),
                        "key_fields_json": json.dumps(key_fields, ensure_ascii=False),
                        "line": line_no,
                    }

                    key = (source, line_no)
                    if key in seen: continue
                    seen.add(key)

                    doc = Document(page_content=content, metadata=metadata)
                    docs.append(doc)
                except Exception as e:
                    print(f"[第 {line_no} 行] 解析失败: {e}")

        print(f"加载 {len(docs)} 条文档")
        return docs

    # ------------------- 添加文档（手动实现） -------------------
    def add_documents(self) -> int:
        raw_docs = self._load_documents()
        if not raw_docs:
            raise ValueError("无文档")

        child_docs = []
        parent_entries = []

        for doc in raw_docs:
            file_id = doc.metadata["file_id"]
            # 切小块
            chunks = self.child_splitter.split_documents([doc])
            for i, chunk in enumerate(chunks):
                child_id = f"{file_id}_chunk_{i}"
                chunk.metadata["parent_id"] = file_id
                chunk.metadata["chunk_index"] = i
                child_docs.append((child_id, chunk))

            # 存大文档（pickle 序列化）
            parent_entries.append((file_id, pickle.dumps(doc)))

        # 写入向量库（小块）
        ids, docs = zip(*child_docs)
        self.vector_store.add_documents(docs, ids=ids)

        # 写入 LocalFileStore（大文档）
        self.docstore.mset(parent_entries)

        print(f"成功写入 {len(raw_docs)} 条大文档（{len(child_docs)} 个小块）")
        return len(raw_docs)

    # ------------------- 增量添加单条 -------------------
    def add_single(self, content: str, metadata: dict = None) -> str:
        if metadata is None:
            metadata = {}
        file_id = metadata.get("file_id", str(uuid.uuid4()))
        metadata["file_id"] = file_id
        doc = Document(page_content=content, metadata=metadata)

        # 切小块
        chunks = self.child_splitter.split_documents([doc])
        child_docs = []
        for i, chunk in enumerate(chunks):
            child_id = f"{file_id}_chunk_{i}"
            chunk.metadata["parent_id"] = file_id
            child_docs.append((child_id, chunk))

        # 写入向量库
        ids, docs = zip(*child_docs)
        self.vector_store.add_documents(docs, ids=ids)

        # 写入大文档
        self.docstore.mset([(file_id, pickle.dumps(doc))])

        print(f"增量添加成功: {file_id}")
        return file_id

    # ------------------- 检索 -------------------
    def search(self, query: str, top_k: int = TOP_K) -> List[Dict[str, Any]]:
        # 检索小块
        results = self.vector_store.similarity_search_with_score(query, k=top_k * 3)
        parent_ids = set()
        for doc, _ in results:
            parent_id = doc.metadata.get("parent_id")
            if parent_id:
                parent_ids.add(parent_id)

        # 取前 top_k 个大文档
        parent_ids = list(parent_ids)[:top_k]
        parent_bytes = self.docstore.mget(parent_ids)
        docs = [pickle.loads(b) for b in parent_bytes if b is not None]

        formatted = []
        for doc in docs:
            formatted.append({
                "content": doc.page_content,
                "metadata": doc.metadata,
            })
        print(f"检索到 {len(formatted)} 条完整凭证")
        return formatted
    
manager = VectorStoreManager()
manager.add_documents()

# 增量添加
manager.add_single(
    content="新凭证...\n关键字段：日期: 2025-11-04",
    metadata={"source": "manual/new.txt", "label": "test"}
)

# 检索
results = manager.search("日期: 2025-11-04")
print(results[0]["content"])


正在初始化 Qwen Embedding 模型...
向量库 + 大文档存储初始化完成
加载 100 条文档


InvalidKeyException: Invalid key: c4a8a5b6-1501-46c3-9266-db1f69a64201. Key should be relative to the full path./data/postgraduates/2024/chenjiarui/Model/Agent/test_tool/../data/parent_docs vs. /data/postgraduates/2024/chenjiarui/Model/Agent and full path of /data/postgraduates/2024/chenjiarui/Model/Agent/data/parent_docs/c4a8a5b6-1501-46c3-9266-db1f69a64201

成功加载 10 条 label 解释
共加载 100 条有效文档
这是一张由江南农村商业银行出具的特种转账借方传票，用于记录一笔转账交易。凭证上显示了付款单位和收款单位的名称、账号、开户银行以及转账的具体金额和原因。
关键字段：付款单位全称: 江南农村商业银行股份有限公司 | 付款单位账号: 01 | 金额(大写): 壹仟壹佰元整 | 金额(小写): ￥110000 | 转账原因: 季晓玲12月伙食


In [32]:
from langchain_chroma import Chroma


embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    base_url=BASE_URL,
    api_key=API_KEY,
)

db = Chroma(
    persist_directory=PERSIST_DIR,
    collection_name="doc_rag",
    embedding_function=embeddings,
)

print("集合大小:", db._collection.count())


集合大小: 5040


In [33]:
import json
import os
from typing import List, Dict, Any
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
import uuid
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore  # 新增

PERSIST_DIR = "/data/postgraduates/2024/chenjiarui/Model/Agent/script/rag/data/chroma_db_final"
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
BASE_URL = "https://api.siliconflow.cn/v1"
API_KEY = os.getenv("SILICONFLOW_API_KEY", "sk-tgprnspwkhliprfcuobqpfiiwjawxkgaldpfkjtovpfudpmf")
JSONL_BASE = "/data/postgraduates/2024/chenjiarui/Model/Agent/script/rag/data/knowledge_base.jsonl"
JSONL_CLASS = "/data/postgraduates/2024/chenjiarui/Model/Agent/script/rag/data/knowledge_class.jsonl"
PERSIST_DIR = "../data/chroma_db_final" 

EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
BASE_URL = "https://api.siliconflow.cn/v1"
API_KEY = os.getenv("SILICONFLOW_API_KEY", "sk-tgprnspwkhliprfcuobqpfiiwjawxkgaldpfkjtovpfudpmf")

CHUNK_SIZE = 600
CHUNK_OVERLAP = 120
TOP_K = 5


embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    base_url=BASE_URL,
    api_key=API_KEY,
)

vector_store = Chroma(
    persist_directory=PERSIST_DIR,
    embedding_function=embeddings,
    collection_name="doc_rag",  # ⚠️ 一定要和原来的一致
)


# 小块切分器（用于精确检索）
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=80,   # 更小，适合你的字段长度
    chunk_overlap=15,
    separators=["。", "，", " | ", " "]
)

retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    docstore=InMemoryStore(),  # 如果你没存 docstore，可空
    child_splitter=child_splitter,
)

   # ------------------- 加载 label 解释 -------------------
def _load_label_intro() -> Dict[str, str]:
    intro = {}
    if not os.path.exists(JSONL_CLASS):
        print(f"警告: 未找到 {JSONL_CLASS}，将跳过 label 解释")
        return intro
    with open(JSONL_CLASS, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                obj = json.loads(line)
                k, v = list(obj.items())[0]
                intro[k] = v
            except:
                continue
    print(f"成功加载 {len(intro)} 条 label 解释")
    return intro

# ------------------- 加载主文档 -------------------
def _load_documents() -> List[Document]:
    docs: List[Document] = []
    seen = set()
    label_intro = _load_label_intro()

    if not os.path.exists(JSONL_BASE):
        raise FileNotFoundError(f"未找到主文件: {JSONL_BASE}")

    with open(JSONL_BASE, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line: continue
            try:
                data = json.loads(line)
                vlm = data.get("VLM_text", {})
                if isinstance(vlm, str):
                    vlm = json.loads(vlm)

                # 提取关键字段
                key_fields = vlm.get("key_fields", {})
                summary = vlm.get("content_summary", "").strip()

                # 构建完整内容：摘要 + 关键字段
                field_text = " | ".join(
                    f"{k}: {v}" for k, v in key_fields.items() if v and str(v).strip()
                )
                content = f"{summary}"
                if field_text:
                    content += f"\n关键字段：{field_text}"

                if not content.strip():
                    continue

                # 元数据
                label = data.get("label", "")
                source = data.get("image_path", "").split("data/")[-1]
                metadata = {
                    "file_id": str(uuid.uuid4()),
                    "source": source,
                    "label": label,
                    "label_introduction": label_intro.get(label, '未知文档类型'),
                    "key_fields_json": json.dumps(key_fields, ensure_ascii=False),
                    "line": line_no,
                }

                # 去重
                key = (source, line_no)
                if key in seen: continue
                seen.add(key)

                # 关键：大块用原始 content
                docs.append(Document(page_content=content, metadata=metadata))

            except Exception as e:
                print(f"[第 {line_no} 行] 解析失败: {e}")

    print(f"共加载 {len(docs)} 条有效文档")
    return docs

docs = _load_documents()
retriever.add_documents(docs)

print(vector_store._collection.count())  # 打印向量条目数

results = vector_store.similarity_search("季晓玲12月伙食", k=3)
for i, doc in enumerate(results, 1):
    print(f"\n--- [{i}] ---")
    print(doc.page_content)

成功加载 10 条 label 解释
共加载 100 条有效文档
5400

--- [1] ---
| 转账原因: 季晓玲12月伙食

--- [2] ---
| 转账原因: 季晓玲12月伙食

--- [3] ---
| 转账原因: 季晓玲12月伙食
